# Fourier Pricing

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Purpose:** Demonstrate the Fourier-based option pricers exposed by `finstack_quant.valuations` and compare them to familiar Black-Scholes intuition.

**Prerequisites:** Basic option pricing, put-call parity, and the difference between diffusion and jump or pure-jump models.

**In this notebook:** We compare COS pricing under Black-Scholes, then extend the same framework to Variance Gamma and Merton jump-diffusion examples.


## Concept

Fourier pricers are useful because they let you evaluate several models through a similar interface once you know the characteristic function. Here we use them for three jobs:

1. Verify numerical agreement against Black-Scholes.
2. Compare smile behavior under a richer model.
3. Check parity and sanity conditions.


In [ ]:
import math

import numpy as np

from finstack_quant.valuations import (
    bs_cos_price,
    merton_jump_cos_price,
    vg_cos_price,
)


def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_closed_form(
    spot: float,
    strike: float,
    rate: float,
    dividend: float,
    vol: float,
    maturity: float,
    is_call: bool,
) -> float:
    if maturity <= 0.0 or vol <= 0.0:
        return max(spot - strike, 0.0) if is_call else max(strike - spot, 0.0)
    fwd = spot * math.exp((rate - dividend) * maturity)
    sqrt_t = math.sqrt(maturity)
    d1 = (math.log(fwd / strike) + 0.5 * vol * vol * maturity) / (vol * sqrt_t)
    d2 = d1 - vol * sqrt_t
    df = math.exp(-rate * maturity)
    if is_call:
        return df * (fwd * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - fwd * norm_cdf(-d1))


def check_parity(pricer, spot, strike, rate, dividend, maturity, *extra):
    call = pricer(spot, strike, rate, dividend, *extra, maturity, True)
    put = pricer(spot, strike, rate, dividend, *extra, maturity, False)
    forward_disc = spot * math.exp(-dividend * maturity) - strike * math.exp(-rate * maturity)
    return call, put, (call - put) - forward_disc


## Model comparison

The next cell follows the original script closely: a Black-Scholes cross-check, a Variance Gamma smile, a Merton jump grid, and parity residuals. The goal is not just to print prices, but to show the kinds of invariants you would want to test whenever you add a new pricing method.


In [ ]:
spot = 100.0
rate = 0.05
dividend = 0.02
maturity = 1.0
bs_vol = 0.20
strike_atm = 100.0

cf_price = bs_closed_form(spot, strike_atm, rate, dividend, bs_vol, maturity, True)
cos_price = bs_cos_price(spot, strike_atm, rate, dividend, bs_vol, maturity, True)
print("Black-Scholes ATM call")
print(f"  closed form : {cf_price:.6f}")
print(f"  COS         : {cos_price:.6f}")
print(f"  COS error   : {abs(cos_price - cf_price):.2e}")

strikes = np.array([80.0, 90.0, 95.0, 100.0, 105.0, 110.0, 120.0])
vg_sigma, vg_theta, vg_nu = 0.20, -0.10, 0.20
m_sigma, m_lambda, m_mu_j, m_sigma_j = 0.20, 0.50, -0.10, 0.15

vg_calls = np.array([
    vg_cos_price(spot, float(k), rate, dividend, vg_sigma, vg_theta, vg_nu, maturity, True)
    for k in strikes
])

merton_calls = np.array([
    merton_jump_cos_price(
        spot,
        float(k),
        rate,
        dividend,
        m_sigma,
        m_mu_j,
        m_sigma_j,
        m_lambda,
        maturity,
        True,
    )
    for k in strikes
])

print("\nVariance Gamma call smile")
for strike, price in zip(strikes, vg_calls):
    print(f"  K={strike:>6.2f} -> {price:.6f}")

print("\nMerton jump-diffusion call grid")
for strike, price in zip(strikes, merton_calls):
    print(f"  K={strike:>6.2f} -> {price:.6f}")

parity_strike = 105.0
for label, pricer, extra in (
    ("BS/COS", bs_cos_price, (bs_vol,)),
    ("VG/COS", vg_cos_price, (vg_sigma, vg_theta, vg_nu)),
    ("Merton/COS", merton_jump_cos_price, (m_sigma, m_mu_j, m_sigma_j, m_lambda)),
):
    _, _, residual = check_parity(pricer, spot, parity_strike, rate, dividend, maturity, *extra)
    print(f"\n{label:<12} parity residual: {residual:.2e}")


## Takeaways

- `bs_cos_price()` is a natural regression check because they should agree with closed-form Black-Scholes.
- `vg_cos_price()` and `merton_jump_cos_price()` show how richer distributions change strike-by-strike prices without changing the overall workflow.
- Parity checks are cheap and valuable whenever you build a new pricing notebook or add a new model binding.


In [ ]:
{
    "bs_closed_form": round(cf_price, 6),
    "bs_cos": round(cos_price, 6),
    "vg_call_min": round(float(vg_calls.min()), 6),
    "merton_call_max": round(float(merton_calls.max()), 6),
}


## Closed-form primitives (BS + exotics)

The root module also exposes direct Black-Scholes and closed-form exotic pricers for quick checks (independent of full instrument JSON).


In [ ]:
from finstack_quant.valuations import bs_price, asian_option_price, barrier_call

print("BS call:", round(bs_price(100, 100, 0.05, 0.01, 0.20, 1.0, True), 6))
print("Asian arith:", round(asian_option_price(100, 100, 0.05, 0.0, 0.20, 1.0, 4, "arithmetic", True), 6))
print("Barrier down-out:", round(barrier_call(100, 100, 90, 0.05, 0.0, 0.20, 1.0, "down", "out"), 6))
